# Sales Analyst 3B Fine-Tune (QLoRA)

**Base model**: `Qwen/Qwen2.5-Coder-3B-Instruct`  
**Hardware**: Colab Pro, A100 GPU (~90 min)  
**Config**: `training/config/lora_3b.yaml`  
**Data**: `data/v1/train.jsonl`  
**Output**: `models/adapters/3b/`

Identical structure to `finetune_1.5b.ipynb` — only the config file differs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q peft==0.10.0 transformers==4.40.2 trl==0.8.6 bitsandbytes==0.43.1 accelerate==0.29.3 datasets==2.18.0 sentencepiece==0.2.0 pyyaml

In [ ]:
import json
import os
import sys
from pathlib import Path

import yaml
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/salestools-analyst")

# Only difference from 1.5B notebook: CONFIG_PATH and ADAPTER_OUT
CONFIG_PATH   = REPO_ROOT / "training/config/lora_3b.yaml"
PROMPT_PATH   = REPO_ROOT / "training/config/system_prompt.txt"
TRAIN_DATA    = REPO_ROOT / "data/v1/train.jsonl"
ADAPTER_OUT   = REPO_ROOT / "models/adapters/3b"

ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()
print("Config:", cfg)

In [ ]:
CHATML_TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n{user}<|im_end|>\n"
    "<|im_start|>assistant\n{assistant}<|im_end|>"
)

records = []
with open(TRAIN_DATA) as f:
    for line in f:
        pair = json.loads(line.strip())
        if not pair.get("verified", False):
            continue
        text = CHATML_TEMPLATE.format(
            system=SYSTEM_PROMPT,
            user=pair["question"],
            assistant=pair["code"],
        )
        records.append({"text": text})

dataset = Dataset.from_list(records)
print(f"Loaded {len(dataset)} verified training pairs")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg["base_model"],
    max_seq_length=cfg["max_seq_len"],
    dtype=None,
    load_in_4bit=True,
)
print("Base model loaded.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora_rank"],
    target_modules=cfg["target_modules"],
    lora_alpha=cfg["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["seed"],
    use_rslora=False,
    loftq_config=None,
)
print("LoRA applied.")
model.print_trainable_parameters()

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=cfg["max_seq_len"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation"],
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=cfg["seed"],
        output_dir=str(ADAPTER_OUT / "checkpoints"),
        save_steps=100,
        save_total_limit=2,
        report_to="none",
    ),
)
print("Trainer ready.")

In [ ]:
trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"Adapter saved to {ADAPTER_OUT}")

## Next Steps

1. Download `models/adapters/3b/` from Drive
2. Run `MODEL_SIZE=3b ADAPTER_PATH=models/adapters/3b/ bash training/export.sh`
3. Eval: `python eval/run_eval.py --model sales-analyst-3b --held-out data/v1/held_out.jsonl`
4. Compare: `python eval/compare.py eval/reports/sales-analyst-1.5b-*.json eval/reports/sales-analyst-3b-*.json`